# importing the necesarry libraries and function

In [ ]:
import os                                                                                   # OperationSystem : libraries, files, paths
import cv2                                                                                  # OpenCV modul : image processing
import numpy as np                                                                          # Numerical Python : calculatons
from skimage.feature import local_binary_pattern                                            # lbp feature calculation
from concurrent.futures import ThreadPoolExecutor                                           # for multithreaded working
from sklearn.metrics import f1_score, balanced_accuracy_score, classification_report        # for showing the result
import joblib                                                                               # for saving the model
from sklearn.ensemble import HistGradientBoostingClassifier                                 # the model type which we will use
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV                     # for training and parameter optimization



# defining the image cutting function

In [ ]:
# to cut up images into smaller pieces from one pre-defined directory to another

def cutting():

    # going into the images folder, from the base_path (thats contains the masked images)
    os.chdir(r"\folder\that\has\edited\images")
    
    
    # for each image in the edited image directory, we do the following
    for image_name_precut in os.listdir():
        # load the image based on it's name
        whole_image = cv2.imread(image_name_precut)
        
        # using median blur to lower noise, while keeping the details, using 9 as it's the highest noise reduction, that dont sacrifice quality
        whole_image_median = cv2.medianBlur(whole_image, 9)
        
        # square x and y dimensions in pixels
        pixel_x = 64
        pixel_y= 64

        # to calcualte how many 64x64 square can we fit in horizontally and vertiaccly
        horizontal_count = whole_image_median.shape[1] // pixel_x
        vertical_count = whole_image_median.shape[0] // pixel_y
        
        # for the same image, each square is identified by cut_it 
        cut_id = 1
        
        # the starting and ending pixel determineation for each cycle horizontally
        for x in range(horizontal_count):
            x_range_start = x*pixel_x
            x_range_end = pixel_x + x*pixel_x

            # for each horizontal count the starting and ending pixel determineation vertically
            for y in range(vertical_count):
                y_range_start = y*pixel_y
                y_range_end = pixel_y + y*pixel_y
                            
                # determining the square by assigning the original images value inside a given boundary
                cut_image = whole_image_median[y_range_start:y_range_end, x_range_start:x_range_end]
                
                # saving the square if there is zero fully black, 0/0/0 HSV value pixel in it
                if cut_image.min() != 0:
                    os.chdir(r"\path\where\to\save\the\cutted\images")
                    cut_image_name = f"{image_name_precut.replace('.jpg', '')}-{cut_id}.jpg"
                    cv2.imwrite(cut_image_name, cut_image) 
                    
                    # id increase by 1
                    cut_id += 1

                    # return back to the edited images directory
                    os.chdir(r"\folder\that\has\edited\images")

# doing the cutting

In [ ]:
cutting()

# defining the category extractor function

In [3]:
# im_name: the name of the image which we want to categorize
def categorize(im_name):
    
    # splitting the image's name into 2 parts, timeofday (day, night, dimness (sunrise or dusk)) and partofsky (cloud, sky, other)
    timeofday = im_name.split("-")[0]
    partofsky = im_name.split("-")[1]
    
    # returning the category number (integer) based on timeofday and partofsky combinations, 9 types total
    if timeofday == "day":
        if partofsky == "sky":
            return(0)
        elif partofsky == "cloud":
            return(1)
        elif partofsky == "other":
            return(2)

    elif timeofday == "night":
        if partofsky == "sky":
            return(3)
        elif partofsky == "cloud":
            return(4)
        elif partofsky == "other":
            return(5)

    elif timeofday == "dimness":
        if partofsky == "sky":
            return(6)
        elif partofsky == "cloud":
            return(7)
        elif partofsky == "other":
            return(8)

# defining the feature extractor functions

In [ ]:
# every feature is scaled between 0 and 180

# Hue lookup tables.
_HUE_ANGLES = np.arange(180, dtype=np.float32) * (2.0 * np.pi / 180.0)
_HUE_SIN = np.sin(_HUE_ANGLES)
_HUE_COS = np.cos(_HUE_ANGLES)
_HUE_SCALE = 90.0 / np.pi

# Saturation and Value lookup tables
_SV_MEAN_SCALE = 180.0 / 255.0
_SV_STD_SCALE = 180.0 / 127.5



# how to get the hue features
def get_hue_features(im_hsv):

    # getting the hue original value and saturation value between 0-1 (for weights)
    hue = im_hsv[:, :, 0].astype(np.intp, copy=False)
    sat = im_hsv[:, :, 1].astype(np.float32, copy=False) / 255.0

    # using the saturation weight to calculate the sum of angles
    sin_sum = np.sum(sat * _HUE_SIN[hue], dtype=np.float32)
    cos_sum = np.sum(sat * _HUE_COS[hue], dtype=np.float32)
    sat_sum = np.sum(sat, dtype=np.float32)

    # return 0 for hue mean and 180 for stddev, when saturation is almost zero.
    if sat_sum <= 1e-6:
        return 0.0, 180.0

    # using arctan2 to keep quadrant information, 
    # getting the direction of hue in degree (using hue_scale table), 
    # using % 180 to warp the value back to 1 instead of 181
    hue_mean = (np.arctan2(sin_sum, cos_sum) * _HUE_SCALE) % 180.0

    # r is the normalized (by sat_sum) lenght of the vector (hue_mean), 
    # forcing it between 0 and 1 by clip() as log(0) is undefined and lenght of R should be 0-1 anyway
    r = np.sqrt(sin_sum * sin_sum + cos_sum * cos_sum) / sat_sum
    r = float(np.clip(r, 1e-6, 1.0))

    # if R is 1, the value is 0 (hues are identical), 
    # if hue is more diverse, R become less then 1, log(r) will become increasing negative value, the sttdev is increasing
    hue_std = np.sqrt(-2.0 * np.log(r)) * _HUE_SCALE

    return float(hue_mean), float(hue_std)



# how to get the saturation/value features
def get_saturation_value_features(im_hsv):

    # get the mean and standard deviation of the given HSV image
    mean, std = cv2.meanStdDev(im_hsv)

    # get the mean of channel 1 and 2 which is saturation and value, 
    # values are converted to a 180 scale
    sat_mean = mean[1, 0] * _SV_MEAN_SCALE
    val_mean = mean[2, 0] * _SV_MEAN_SCALE

    # get the stddev of channel 1 and 2 which is saturation and value, 
    # values are converted to a 180 scale
    sat_std = std[1, 0] * _SV_STD_SCALE
    val_std = std[2, 0] * _SV_STD_SCALE

    return sat_mean, val_mean, sat_std, val_std



# how to get the lbp histogram feature
def get_lbp_features(im_gray):

    # using grayscale version (only intrested in texture, not color), 
    # with 8 neighbor pixels, 
    # 1 radius,
    # uniform method are all choosen by trail and error 
    lbp = local_binary_pattern(im_gray, 8, 1, method="uniform")

    # histogram shows how frequantly a value occures, 
    # ravel turn the image values into a list, 
    # bin is the category number, 
    # range means 0-10, 
    # densit=true normalize the histogram, so wont be effected by image size
    hist, _ = np.histogram(lbp.ravel(), bins=10, range=(0, 10), density=True)

    # normalizing the value to 180 scale
    hist = hist.astype(np.float32) * 180.0

    return hist



# to describe how to get the category and features
def get_category_feature_array(image_path, image_name):

    # to read the image
    im = cv2.imread(image_path)

    # to get the grayscale and HSV version of the image for LBP and HSV features
    im_gray = cv2.cvtColor(im, cv2.COLOR_BGR2GRAY)
    im_hsv = cv2.cvtColor(im, cv2.COLOR_BGR2HSV)

    # to get the normalized mean and stddev of hue
    hue_mean, hue_std = get_hue_features(im_hsv)

    # to get the normalized mean and stddev of saturation and value
    sat_mean, val_mean, sat_std, val_std = get_saturation_value_features(im_hsv)

    # to get the normalized lbp histogram
    lbp_features = get_lbp_features(im_gray)

    # to get the image category, can be 0 to 8
    category = np.uint8(categorize(image_name))

    # to make a color features array of float32 (required format for sklearn training is either float32 or float64)
    color_features = np.array([hue_mean, sat_mean, val_mean, hue_std, sat_std, val_std], dtype=np.float32)

    # the final feature array is made of color and lbp features combined
    feature = np.concatenate((color_features, lbp_features))

    return category, feature



# how to do the extraction faster
def get_folder_categ_features(folder, max_workers):

    # we make lists out of the image's names and their paths
    image_names = []
    image_paths = []

    for name in os.listdir(folder):
        if name.endswith(".jpg"):
            image_names.append(name)
            image_paths.append(os.path.join(folder, name))

    # we make lists for the calculated categories and their features, same index means same the category and feature is related to the same image
    categ_list = []
    feature_list = []

    # to make the work multithreaded (incredibly faster, more than 90 percent reduction in my case), 
    # getting the category and feature, adding them to a list for each given image based on it's name and path (from the previous 2 list),
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        for category, feature in executor.map(get_category_feature_array, image_paths, image_names):
            categ_list.append(category)
            feature_list.append(feature)

    # the full category and feature pair list for a given folder
    categ_array = np.asarray(categ_list, dtype=np.uint8)
    feature_array = np.asarray(feature_list, dtype=np.float32)

    return categ_array, feature_array



# to do the extraction of the features and categories for a given folder
def get_train_test_categ_features(train_dir, test_dir):

    # workers (threads) count, the cpu core count (or 1 if cant determine) + 4, i have 8+4 so 12 threads out of total 16 (it can be maximum 32)
    max_workers = min(32, (os.cpu_count() or 1) + 4)

    # process and get the category and features for train and test dataset
    train_categ, train_features = get_folder_categ_features(train_dir, max_workers)
    test_categ, test_features = get_folder_categ_features(test_dir, max_workers)

    return train_categ, train_features, test_categ, test_features

# training the model, printing out the parameters and  results and saving the best model

In [ ]:
# get the train, test category and feature arrays for a given base path
train_categ, train_features, test_categ, test_features = get_train_test_categ_features(r"\path\for\the\cutted\training\images", r"\path\for\the\cutted\test\images")

# where to save the model
save_path = r"path\for\the\trained\models"

# Test distributionaware class weights (knowing the imballance of train category distribution)
class_weight_dict = {0: 0.00065, 1: 0.15, 2: 19.00, 3: 0.02, 4: 0.15, 5: 5.70, 6: 0.40, 7: 1.0, 8: 0.90}

# using a 5 fold stratified cross validaton (trying to keep the origianl data category ratio for the split)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)

# using histogram gradient boost decision tree classifier, it caputres well complex relations of features and can handle multiple classes, (much better than my original linear SVM model)
#  using the class weights,
#  300 maximum iterations (total amount of trees is max_iter * classes (9 in my case)), 
# early stopping if no improvment, 
# 10% of training data used for validaition of early stopping, 
# if in 20 iteration not improve > stop, 
# verbose=0 means dont print a lot of training inforamtion 
model = HistGradientBoostingClassifier(class_weight=class_weight_dict, max_iter=300, early_stopping=True, validation_fraction=0.1, n_iter_no_change=20, random_state=0, verbose=0,)

# hyperparameter search (values got from trial and error), (this is my last training, so has 1 value for each parameter, for finding multiple parameters, i used 3 values for each parameter so 81 total combination)
# min leaf restricts the minimum data for training sample (only can be grater), usefull to avoid 1/1000 decisions, which is most of the time noise
# depth adjust the amount of decision depth, more depth means more complex relation, but also makes the model more prone to overfitting
# learning rate how strong each tree contribute to final model, faster rate gave each tree more weight, but risk overfitting, recommened to use earyly_stopping to optimize training
# regularization discourage the model becoming overfitted by applying a penalti to the models leaf values
param_distributions = {
    "min_samples_leaf": [30],
    "max_depth": [16],
    "learning_rate": [0.06],
    "l2_regularization": [1]
}


# try to find the best hyperparameters combination, 
# model is what we want to tune (HGBDT), 
# paramdistribion is the possible values for the parameters, 
# niter chose n combination out of the total randomly, must be less than the total amount of parameter combination
# cv is the cross validation which it uses (StratifiedKfold)
# f1 macro (average every class) will be the scoring, f1 score is prefered for imballanced datasets
# njobs -1 make use of all cpu cores for the search
# refit=true will train the final model of the full training data using the best hyperparameters
search = RandomizedSearchCV(estimator=model, param_distributions=param_distributions, n_iter=1, cv=cv, scoring="f1_macro", n_jobs=-1, refit=True, random_state=0,)

# starts the training using the training features and categories
search.fit(train_features, train_categ)

# print out the best parameters and the best macro f1 achived
print("\nBest params:", search.best_params_)
print("Best CV macro F1:", search.best_score_)

# saves the model using the best hyperparameters, same as refit=ture
best_model = search.best_estimator_

# saves the best models macro f1 score on the training data
best_cv_macro_f1 = search.best_score_

# make a prediction on the test data using the best model and the test features
pred = best_model.predict(test_features)

# this calculates the f1 score on the test data
test_macro_f1 = f1_score(test_categ, pred, average="macro")

# calculates the ballanced accuracy of test data prediction using recall (correctly predicted positives (True positives) and total positives (True positive + False negatives) ratio)
test_balanced_acc = balanced_accuracy_score(test_categ, pred)

# prining out the results
print("\n--- Test Classification Report ---")
print(classification_report(test_categ, pred, digits=3))
print(f"Test Macro F1: {test_macro_f1:.4f}")
print(f"Test Balanced Acc: {test_balanced_acc:.4f}")
print(f"CV/Test Gap: {best_cv_macro_f1 - test_macro_f1:.4f}")

# creating the model name 
model_name = os.path.join(save_path, "your_model_name.joblib")

# saving the best model with the created name
joblib.dump(best_model, model_name)

# print, that the model is saved
print("Saved:", model_name)